# Real World Validation

Goal:
Evaluate whether the baseline model generalizes.

Dataset:
Custom recordings

Questions:

- Does model survive outside ESC50?
- Which classes fail?

In [ ]:
from pathlib import Path

import joblib
import librosa
import numpy as np
import pandas as pd
import tensorflow_hub as hub

In [ ]:
clf = joblib.load("../models/baseline.pkl")

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")

## Predict Sound

In [ ]:
def predict_audio(path):

    waveform, sr = librosa.load(path, sr=16000)

    waveform = waveform.astype(np.float32)

    scores, embeddings, spec = yamnet(waveform)

    emb = embeddings.numpy().mean(axis=0)

    pred = clf.predict([emb])[0]

    conf = clf.predict_proba([emb]).max()

    return pred, conf

In [ ]:
sample = "../data/real_world/audio/crying_baby/baby_2.wav"

In [ ]:
predict_audio(sample)

## Evaluate Folder

In [ ]:
ROOT = Path("../data/real_world/audio")

results = []

In [ ]:
for label in ROOT.iterdir():
    for file in label.glob("*.wav"):
        pred, conf = predict_audio(file)

        results.append({"true": label.name, "pred": pred, "confidence": conf})

In [ ]:
df = pd.DataFrame(results)

df

In [ ]:
(df["true"] == df["pred"]).mean()

In [ ]:
df[df["true"] != df["pred"]]

# Findings

## Real World Accuracy

0.9487 (94.87%)

Result:
Performance remained strong outside the training dataset.

---

## Observations

Most sounds generalized correctly.

A small number of recordings failed.

Expected degradation occurred compared to cross validation.

Cross Validation:
99.38%

Real World:
94.87%

Gap:
≈ 4.5%

---

## Interpretation

The model appears to learn sound characteristics rather than memorizing ESC50.

Performance is promising for a first baseline.

---

## Limitations

Validation size remains small.

Real environments are still limited.

Long-term robustness unknown.

---

## Decision

Accept baseline.

Proceed toward deployment preparation.